# LexChain chunking × embedding retrieval benchmark

3 chunkers (LangChain `RecursiveCharacterTextSplitter`, Chonkie `RecursiveChunker`, LlamaIndex `SentenceSplitter`; ~512 tok / ~50 overlap, cl100k_base counting) × 3 embedders (`e5-base-v2`, `bge-base-en-v1.5`, `all-MiniLM-L6-v2`) on **OHR-Bench Law**: 95 GT docs, 1,142 QAs. Evidence matching mirrors OHR-Bench's doc+page gate + word-LCS.

**Runtime:** free T4 GPU (`Runtime → Change runtime type → T4 GPU`).

Everything is cached to **Google Drive** (`MyDrive/lexchain_bench/`) after every stage — a disconnect loses nothing; rerun the cell and it resumes.

In [ ]:
# Cell 1 — mount Drive, clone repo, install pinned deps (torch stays Colab's build)
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/lexchain-chunk-embed-bench'):
    !git clone https://github.com/MarvinPescos/lexchain-chunk-embed-bench.git /content/lexchain-chunk-embed-bench
%cd /content/lexchain-chunk-embed-bench
!git pull
!pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable the T4 runtime!')

In [ ]:
# Cell 2 — download OHR-Bench Law ground truth + QA pairs (idempotent)
!python download_data.py

In [ ]:
# Cell 3 — smoke test: full 9-config matrix on 5 docs, then project the full run
!python bench.py --limit-docs 5
!python aggregate.py --suffix _n5

import json, glob
from pathlib import Path

gt = {p: json.loads(Path(p).read_text()) for p in glob.glob('data/gt/law/*.json')}
chars = {Path(p).stem: sum(len(pg.get('text', '')) for pg in pages) for p, pages in gt.items()}
smoke_docs = sorted(chars)[:5]
factor = sum(chars.values()) / max(sum(chars[d] for d in smoke_docs), 1)
smoke = [json.loads(Path(p).read_text()) for p in glob.glob('/content/drive/MyDrive/lexchain_bench/results/*_n5.json')]
embed_s = sum(r['embed_s'] for r in smoke)
chunk_s = sum({r['chunker']: r['chunk_time_s'] for r in smoke}.values())
est_min = (embed_s + chunk_s) * factor / 60
print(f"\n=== Projection: full run (95 docs) ≈ {est_min:.0f} min of compute "
      f"(smoke×{factor:.0f}; + ~5 min model loads). Colab free tier allows this in one session. ===")

**STOP — check the smoke tables and the projection above before the full run.** If Colab disconnects mid-run, just rerun Cell 4: completed chunk/embedding/result stages are read back from Drive.

In [ ]:
# Cell 4 — full benchmark (95 docs, 1,142 QAs, 9 configs; resumable)
!python bench.py

In [ ]:
# Cell 5 — aggregate: CSVs + chunker/embedder marginal tables + 9-row matrix
# + interaction-effect flags, all written to Drive (MyDrive/lexchain_bench/) and printed
!python aggregate.py